# 03 — Advanced Visualizations
Create reusable plots from actual columns. Replace `revenue` and date fields with verified column names if required.

In [ ]:
import os
import pandas as pd
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

load_dotenv("../.env")
host = os.getenv("PGHOST", "localhost")
port = os.getenv("PGPORT", "5432")
database = os.getenv("PGDATABASE", "ecommerce_practice")
user = os.getenv("PGUSER", "postgres")
password = os.getenv("PGPASSWORD", "")
schema = os.getenv("PGSCHEMA", "day24_gold")

if not password:
    raise ValueError("Set PGPASSWORD in the project-root .env file.")

engine = create_engine(
    f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{database}"
)
print("Connected to PostgreSQL")


In [ ]:
fact = pd.read_sql(text(f'SELECT * FROM "{schema}"."fact_sales"'), engine)
display(fact.head())


## Revenue distribution (if revenue exists)

In [ ]:
import matplotlib.pyplot as plt
import plotly.express as px

if "revenue" in fact.columns:
    fig = px.histogram(fact, x="revenue", nbins=40,
                       title="Revenue Distribution",
                       marginal="box")
    fig.show()
else:
    print("Column 'revenue' not found. Update the column name after schema inspection.")


## Monthly revenue trend
Requires a date column in the fact table. If the fact table only has a date key, join to dim_date in SQL first.

In [ ]:
if {"full_date", "revenue"}.issubset(fact.columns):
    fact["full_date"] = pd.to_datetime(fact["full_date"])
    monthly = (fact.set_index("full_date")["revenue"]
               .resample("MS").sum().reset_index())
    fig = px.line(monthly, x="full_date", y="revenue",
                  markers=True, title="Monthly Revenue")
    fig.show()
else:
    print("No full_date + revenue pair found in fact_sales. Build a SQL join to dim_date.")


## Revenue Pareto (requires product/category field and revenue)

In [ ]:
candidate = next((c for c in ["category", "product_name"] if c in fact.columns), None)
if candidate and "revenue" in fact.columns:
    agg = (fact.groupby(candidate, dropna=False)["revenue"]
           .sum().sort_values(ascending=False).reset_index())
    agg["cumulative_share_pct"] = 100 * agg["revenue"].cumsum() / agg["revenue"].sum()
    fig = px.bar(agg, x=candidate, y="revenue", title="Revenue by " + candidate)
    fig.show()
    display(agg)
else:
    print("Join dim_product to fact_sales to obtain category/product labels.")
